In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


#Clone the repository

In [ ]:
%cd /content/drive/MyDrive/
!git clone https://github.com/IvanXunZhang/SoMe

/content/drive/MyDrive
fatal: destination path 'SoMe' already exists and is not an empty directory.


In [ ]:
# !mv /content/drive/MyDrive/DB/content/drive/MyDrive/SoMe/database

%cd /content/drive/MyDrive/SoMe/

/content/drive/MyDrive/SoMe


In [ ]:
# !pip install torch torchvision torchaudio

#Clone the repository

In [ ]:
%cd /content/SoMe/

!pip install -r requirements.txt

[Errno 2] No such file or directory: '/content/SoMe/'
/content/drive/MyDrive/SoMe


In [ ]:
!pip install dashscope

In [ ]:
%%writefile /content/drive/MyDrive/SoMe/test_misinformation_detection.py
import json
import os
import re
from time import sleep
from datetime import datetime, time, timezone
import argparse

from colorama import *
from tqdm import tqdm

from qwen_agent.utils.output_beautify import typewriter_print
from qwen_agent.llm.base import ModelServiceError
import tasks.misinformation_detection as task
from agent import SocialMediaAgent
from tools.en import *

parser = argparse.ArgumentParser()
parser.add_argument("--model", type=str, default="Meta-Llama-3.1-8B-Instruct", help="The base model for the agent")
parser.add_argument('--base_url', type=str, default="http://0.0.0.0:8007/v1", help="The base url for the model server")
parser.add_argument('--api_key', type=str, default="mysecrettoken123", help="The api key for the model server")
parser.add_argument('--output_path', type=str, default="results/misinformation_detection", help="The output path for the results")
args = parser.parse_args()

DEEPSEEK_OFF_PEAK = 'deepseek' in args.model

def remove_think_tags(text):
    # Use regex to match content between <think> and </think> tags and replace with empty string
    cleaned_text = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL)
    return cleaned_text.strip()

os.makedirs(args.output_path, exist_ok=True)
output_file = os.path.join(args.output_path, f"{args.model.split('/')[-1]}.json")
if os.path.exists(output_file):
    results = json.load(open(output_file))
else:
    results = {}

vllm_config = {
    'model_server': args.base_url,
    'api_key': args.api_key,
    'model': args.model,
}
bot = SocialMediaAgent(system_message=task.en_prompt,
                       function_list=['knowledge_retrieve'],
                       llm=vllm_config)
twitters = json.load(open("datasets/misinformation_detection/ground_truth.json", encoding="utf-8"))
twitters = dict(list(twitters.items())[:50])
for tid in tqdm(twitters):
    if tid in results:
        continue
    messages = []  # Store chat history here
    query = f"Please determine if the twitter '{twitters[tid]['claim']}' is true."
    print(f'{Fore.RED + Style.BRIGHT}用户请求:{Style.RESET_ALL} {query}')
    messages.append({'role': 'user', 'content': query})
    response = []
    response_plain_text = ''
    print(f'{Fore.CYAN + Style.BRIGHT}智能体回应:{Style.RESET_ALL}')
    for retry in range(5):
        if DEEPSEEK_OFF_PEAK:
            while True:
                current_utc_time = datetime.now(timezone.utc).time()
                start_time = time(16, 30)
                end_time = time(0, 30)
                is_between = False
                if start_time <= end_time:
                    is_between = start_time <= current_utc_time <= end_time
                else:
                    is_between = current_utc_time >= start_time or current_utc_time <= end_time
                if not is_between:
                    print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}: Waiting for Deepseek off-peak time")
                    sleep(10)
                else:
                    break
        try:
            for response in bot.run(messages=messages, lang='en'):
                response_plain_text = typewriter_print(response, response_plain_text)
            messages.extend(response)
            print('\n')
            result = remove_think_tags(messages[-1]['content'])
            results[tid] = result
            with open(output_file, "w", encoding='utf8') as f:
                json.dump(results, f, ensure_ascii=False, indent=4)
            break
        # except ModelServiceError:
        #     sleep(60)
        #     continue
        # except:
        #     messages = []
        except ModelServiceError:
            sleep(60)
            continue
        except Exception as e:    # <--- 修改这里
            print(f"\n🚨 捕捉到隐藏报错: {e}\n")    # <--- 加上这一行把错误打印出来
            messages = []  # Store chat history here

            query = f"Please determine if the twitter '{twitters[tid]['claim']}' is true."
            print(f'{Fore.RED + Style.BRIGHT}用户请求:{Style.RESET_ALL} {query}')
            messages.append({'role': 'user', 'content': query})
            response = []
            response_plain_text = ''
            print(f'{Fore.CYAN + Style.BRIGHT}智能体回应:{Style.RESET_ALL}')
            continue
    if tid not in results:
        results[tid] = ""

Overwriting /content/drive/MyDrive/SoMe/test_misinformation_detection.py


run1_baseline_error



In [ ]:
from google.colab import userdata

MY_API_KEY = userdata.get('ALIYUN_API_KEY')

!python test_misinformation_detection.py \
  --model "qwen2.5-7b-instruct" \
  --base_url "dashscope" \
  --api_key {MY_API_KEY}

2026-02-25 05:33:19.071398: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771997599.097290   50767 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771997599.105527   50767 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771997599.127776   50767 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771997599.127834   50767 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771997599.127845   50767 computation_placer.cc:177] computation placer alr

run2_modified_error

In [ ]:
from google.colab import userdata

MY_API_KEY = userdata.get('ALIYUN_API_KEY')

!python test_misinformation_detection.py \
  --model "qwen2.5-7b-instruct" \
  --base_url "dashscope" \
  --api_key {MY_API_KEY}

2026-02-27 02:22:20.532929: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772158940.594369   15184 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772158940.617055   15184 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772158940.681910   15184 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772158940.681980   15184 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772158940.681985   15184 computation_placer.cc:177] computation placer alr

run4_modified_error

In [ ]:
from google.colab import userdata

MY_API_KEY = userdata.get('ALIYUN_API_KEY')

!python test_misinformation_detection.py \
  --model "qwen2.5-7b-instruct" \
  --base_url "dashscope" \
  --api_key {MY_API_KEY}

2026-02-27 03:03:30.643889: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772161410.751623   25109 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772161410.774052   25109 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772161410.887187   25109 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772161410.887292   25109 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772161410.887303   25109 computation_placer.cc:177] computation placer alr

run3_baseline_error

In [ ]:
from google.colab import userdata

MY_API_KEY = userdata.get('ALIYUN_API_KEY')

!python test_misinformation_detection.py \
  --model "qwen2.5-7b-instruct" \
  --base_url "dashscope" \
  --api_key {MY_API_KEY}

2026-02-27 03:55:18.155767: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772164518.211781   37755 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772164518.227482   37755 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772164518.281191   37755 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772164518.281254   37755 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772164518.281260   37755 computation_placer.cc:177] computation placer alr

#Calculation code

In [ ]:
import json
import os
import re

def get_real_path(path):
    if os.path.isdir(path):
        return os.path.join(path, "qwen2.5-7b-instruct.json")
    return path

# Specifically designed for the calculation of hallucination rate in Run 1 and Run 2
def calc_hallucination(file_path):
    real_path = get_real_path(file_path)
    if not os.path.exists(real_path):
        print(f"🚨 Cannot find file: {real_path}")
        return

    results = json.load(open(real_path, "r", encoding="utf-8"))
    total = 50
    hallucination_count = 0

    for tid, text in results.items():
        if not text: continue
        text_lower = text.lower()

        has_guess = bool(re.search(r'(answer|statement|is|therefore|thus|so|verdict|be).{0,30}?(true|false|pants-fire|barely-true|half-true|mostly-true)', text_lower))

        if has_guess:
            hallucination_count += 1

    rate = (hallucination_count / total) * 100
    print(f"📊 Evaluate document: {file_path}")
    print(f" - 🚨 Hallucination Rate: {rate:.1f}% (Forced guess on {hallucination_count}/{total} questions)")
    print("-" * 50)

# 🟢 Accurate rate calculation specifically for Run 3 and Run 4
def calc_accuracy(file_path, gt_file="datasets/misinformation_detection/ground_truth.json"):
    real_path = get_real_path(file_path)
    if not os.path.exists(real_path):
        print(f"🚨 Cannot find file: {real_path}")
        return

    results = json.load(open(real_path, "r", encoding="utf-8"))
    twitters = json.load(open(gt_file, "r", encoding="utf-8"))

    correct = 0
    total = 50
    answered = 0

    for tid, text in results.items():
        if not text: continue
        text_lower = text.lower()

        true_label = str(twitters.get(tid, {}).get("label", "")).lower()

        # Match the correct answer
        is_correct = bool(true_label and true_label in text_lower)

        has_guess = bool(re.search(r'(answer|statement|is|therefore|thus|so|verdict|be).{0,30}?(true|false|pants-fire|barely-true|half-true|mostly-true)', text_lower))

        if has_guess or is_correct:
            # As long as it's not a forced error report, it's considered completed.
            if "insufficient information" not in text_lower and "unable to connect" not in text_lower:
                answered += 1
            if is_correct:
                correct += 1

    tcr = (answered / total) * 100
    acc = (correct / total) * 100
    print(f"📊 Evaluate document: {file_path}")
    print(f" - ✅ Task Completion Rate (TCR): {tcr:.1f}% ({answered}/{total})")
    print(f" - 🎯 Accuracy Rate (Accuracy): {acc:.1f}% ({correct}/{total})")
    print("-" * 50)
# ==========================================
# Start calculating your experimental results:
# ==========================================
print("【Phase 1: Tool Error (Testing Illusion Rate)】")
calc_hallucination("results/run1_baseline_error.json/qwen2.5-7b-instruct.json")
calc_hallucination("results/run2_modified_error.json/qwen2.5-7b-instruct.json")

print("\n【Phase Two: Tools are functioning properly (testing accuracy rate)】")
calc_accuracy("results/run3_baseline_error.json/qwen2.5-7b-instruct.json")
calc_accuracy("results/run4_modified_error.json/qwen2.5-7b-instruct.json")

【Phase 1: Tool Error (Testing Illusion Rate)】
📊 Evaluate document: results/run1_baseline_error.json/qwen2.5-7b-instruct.json
 - 🚨 Hallucination Rate: 20.0% (Forced guess on 10/50 questions)
--------------------------------------------------
📊 Evaluate document: results/run2_modified_error.json/qwen2.5-7b-instruct.json
 - 🚨 Hallucination Rate: 0.0% (Forced guess on 0/50 questions)
--------------------------------------------------

【Phase Two: Tools are functioning properly (testing accuracy rate)】
📊 Evaluate document: results/run3_baseline_error.json/qwen2.5-7b-instruct.json
 - ✅ Task Completion Rate (TCR): 100.0% (50/50)
 - 🎯 Accuracy Rate (Accuracy): 66.0% (33/50)
--------------------------------------------------
📊 Evaluate document: results/run4_modified_error.json/qwen2.5-7b-instruct.json
 - ✅ Task Completion Rate (TCR): 68.0% (34/50)
 - 🎯 Accuracy Rate (Accuracy): 44.0% (22/50)
--------------------------------------------------
